# Nanophotonics simulations: project run file

This notebook gives a single place to see (and optionally reproduce) the results of the three projects in this repository:

| Project | What it does | Methods |
|---|---|---|
| **01_cavity_fdtd_vs_tmm** | Ag / dielectric / Ag microcavity: resonance wavelength, linewidth and Q | 1D FDTD (Meep) checked against an own transfer-matrix (TMM) implementation, with a resolution-convergence study |
| **02_metalens_rcwa_gds** | TiO₂ metalens at 633 nm (40 µm diameter, NA 0.45) | RCWA (grcwa) unit-cell library, phase-map design, angular-spectrum focusing check, GDS export (gdsfactory) |
| **03_vortex_beam_generator** | Metasurface that converts a Gaussian beam into an OAM vortex beam | Geometry parameter search, RCWA pillar library, 3D FDTD verification, inverse design (autograd), freeform clean-up for fabrication, GDS export |

**How to use it**

- By default the notebook only **displays the saved results** in each project's `results/` folder, so it runs in seconds.
- To **re-run the simulations**, set the switches in the *Run switches* cell to `True`. Each script is run from its own `src/` folder in a separate Python process, exactly as `run_all.sh` does.
- Environment: `conda env create -f environment.yml`, then `conda activate nanophotonics-simulations` and start Jupyter from that environment.

## Setup

In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from IPython.display import Image, Markdown, display

REPO = Path.cwd()
if not (REPO / "01_cavity_fdtd_vs_tmm").is_dir():
    raise RuntimeError("Please open this notebook from the top folder of the repository.")

PROJECTS = {
    "01": REPO / "01_cavity_fdtd_vs_tmm",
    "02": REPO / "02_metalens_rcwa_gds",
    "03": REPO / "03_vortex_beam_generator",
}


def run_script(project, script):
    """Run one script from the project's src folder and print its output as it arrives."""
    src = PROJECTS[project] / "src"
    print(f"$ cd {src.relative_to(REPO)} && python {script}", flush=True)
    proc = subprocess.Popen(
        [sys.executable, script], cwd=src,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{script} stopped with exit code {proc.returncode}")


def show_results(project):
    """Display the Markdown summaries and PNG figures saved in the project's results folder."""
    results = PROJECTS[project] / "results"
    for summary in sorted(results.glob("*.md")):
        display(Markdown(f"#### {summary.name}\n\n" + summary.read_text()))
    for figure in sorted(results.glob("*.png")):
        display(Markdown(f"#### {figure.name}"))
        display(Image(filename=str(figure)))


import meep
print(f"Python {sys.version.split()[0]}")
print(f"meep {meep.__version__}")
for package in ["numpy", "scipy", "matplotlib", "grcwa", "autograd", "gdsfactory"]:
    try:
        print(f"{package} {version(package)}")
    except PackageNotFoundError:
        print(f"{package} (version not found)")

## Run switches

`False` = show the saved results only. `True` = re-run that part first (this overwrites the files in `results/`).

In [ ]:
RUN_01 = False        # about 1 min: 1D FDTD at several resolutions + TMM comparison
RUN_02 = False        # a few min: RCWA library, metalens design, GDS export
RUN_03_FAST = False   # several min: parameter search, pillar library, freeform clean-up, vortex beam, GDS
RUN_03_SLOW = False   # about 80 min: 3D FDTD of the unit cells (~45 min) and inverse design (~35 min)

## 01: Metal–dielectric–metal microcavity, FDTD vs TMM

A dielectric layer between two thin silver mirrors forms a Fabry–Pérot cavity. The reflection spectrum is computed in two independent ways:

- **TMM**: own transfer-matrix code (`src/tmm.py`), exact for planar layer stacks.
- **FDTD**: 1D Meep simulation at increasing grid resolution.

As the grid is refined, the FDTD resonance wavelength and reflection spectrum converge to the TMM result, which validates both the FDTD set-up and the TMM implementation.

In [ ]:
if RUN_01:
    run_script("01", "run_fdtd.py")
    run_script("01", "compare.py")
show_results("01")

## 02: TiO₂ metalens with RCWA design and GDS export

1. **Unit-cell library** (`unit_cell_library.py`): RCWA simulation of TiO₂ pillars of different diameters (transmission and phase), with checks against TMM for a uniform slab and a Fourier-order convergence study.
2. **Lens design** (`design_metalens.py`): each lattice site gets the pillar whose phase matches the hyperbolic lens profile; the focusing is checked by angular-spectrum propagation (focal position, spot size, Strehl ratio, efficiency).
3. **Layout** (`make_gds.py`): export of all pillars to a GDS file for lithography.

In [ ]:
if RUN_02:
    for script in ["unit_cell_library.py", "design_metalens.py", "make_gds.py"]:
        run_script("02", script)
show_results("02")

## 03: Metasurface vortex-beam (OAM) generator

1. **Parameter search** (`parameter_search.py`): scan of pillar height and period for full 2π phase coverage with high transmission, excluding geometries with a too-high aspect ratio for fabrication.
2. **Pillar library** (`pillar_library.py`): RCWA library for the chosen geometry and selection of the phase levels.
3. **3D FDTD check** (`fdtd_unit_cells.py`): Meep verification of the selected unit cells against RCWA at two resolutions.
4. **Inverse design** (`inverse_design.py`) and **clean-up** (`clean_freeform.py`): gradient-based freeform optimisation, then removal of features that cannot be fabricated.
5. **Vortex beam** (`vortex.py`) and **layout** (`make_gds.py`): spiral phase plate, beam profile and GDS export.

In [ ]:
if RUN_03_FAST:
    run_script("03", "parameter_search.py")
    run_script("03", "pillar_library.py")
if RUN_03_SLOW:
    run_script("03", "fdtd_unit_cells.py")
    run_script("03", "inverse_design.py")
if RUN_03_FAST:
    for script in ["clean_freeform.py", "vortex.py", "make_gds.py"]:
        run_script("03", script)
show_results("03")

## Reproducing everything from the terminal

```bash
conda env create -f environment.yml
conda activate nanophotonics-simulations
./run_all.sh quick   # projects 01 and 02 (~3 min)
./run_all.sh         # all three projects (~1.5 h on 2 cores)
```